# Práctica 10 — Pokémon: Scatter 3D Plot con Sprites
**Programa de Estudios:** Ingeniería en Desarrollo y Gestón de Software \
**Asignatura:** Extracción de Conocimiento en Bases de Datos \
**Docente:** M.T.I Marco A. Ramírez Hernández \
**Periodo:** Mayo - Agosto 2026 \
**Fecha de entrega:**  15 de agosto de 2026

**Objetivo:** construir un dataset de Pokémon (generaciones 1 a 9) a partir de la [PokeAPI](https://pokeapi.co/) y visualizarlo en un **gráfico de dispersión 3D**, usando el sprite de cada Pokémon como símbolo del punto.

**Nombre del Estudiante:** Karen Lizbeth Negrete Hernández \
**Matrícula:** 230570 \
**Grado y Grupo:** 9° B IDGS \
**Repositorio de Git:** https://github.com/karenNegrete06/ECBD_9B_IDGS_Practicas_230570.git

**Variables usadas en el gráfico 3D:**
- Eje X → Generación
- Eje Y → Tipo principal (Type 1)
- Eje Z → Promedio de estadísticas (columna `promedio_estadisticas`: HP, Attack, Defense, Sp. Atk, Sp. Def, Speed)
- Color → Tipo principal
- Símbolo → Sprite del Pokémon

> Este notebook necesita **conexión a internet** para descargar datos e imágenes desde la PokeAPI. La primera ejecución completa (≈1025 Pokémon) puede tardar entre 3 y 8 minutos dependiendo de tu conexión. Los datos se guardan en un CSV local para no tener que repetir la descarga cada vez.


## 1. Librerías necesarias

Si te falta alguna librería, instálala en una celda con, por ejemplo:
```
!pip install requests pandas numpy matplotlib pillow plotly tqdm ipywidgets dash
```

Se importan las librerías necesarias (`requests` para llamar a la PokeAPI, `pandas`/`numpy` para los datos, `matplotlib` y `PIL` para las imágenes, `plotly` para la versión interactiva, `tqdm` para barras de progreso) y se crean las carpetas locales donde se guardarán el dataset (CSV) y los sprites descargados.

In [1]:
import os
import io
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import proj3d
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

from tqdm import tqdm

import plotly.express as px

DATA_DIR = "pokemon_data"
SPRITE_DIR = os.path.join(DATA_DIR, "sprites")
os.makedirs(SPRITE_DIR, exist_ok=True)

CSV_PATH = os.path.join(DATA_DIR, "pokemon_dataset.csv")

## 2. Descarga de datos desde la PokeAPI

Estrategia para minimizar el número de solicitudes:

1. Se consulta el endpoint `/generation/{n}` (n = 1..9) **una sola vez por generación** para saber qué Pokémon (por nombre) pertenece a cada generación. Esto evita tener que pedir la ficha de especie de cada Pokémon individualmente.
2. Se consulta el endpoint `/pokemon/{id}` para cada Pokémon (1 a 1025), de donde se obtienen tipos, estadísticas base y la URL del sprite. Estas solicitudes se hacen en paralelo con `ThreadPoolExecutor` para acelerar la descarga.
3. La condición *Legendary* se determina comparando el nombre contra una lista de Pokémon legendarios/míticos conocidos (Bulbapedia), ya que la PokeAPI solo expone ese dato en el endpoint de especie (que implicaría duplicar las solicitudes).

Se define la función `obtener_generaciones`, que consulta el endpoint `/generation/{n}` de la PokeAPI (una sola vez por cada una de las 9 generaciones) para construir un diccionario que asocia el nombre de cada Pokémon con su número de generación. Esto evita tener que consultar la ficha de especie de cada Pokémon por separado.

In [2]:
BASE_URL = "https://pokeapi.co/api/v2"
N_POKEMON = 1025  # Número nacional de Pokédex cubierto por las generaciones 1-9

ROMAN_TO_GEN = {
    "generation-i": 1, "generation-ii": 2, "generation-iii": 3,
    "generation-iv": 4, "generation-v": 5, "generation-vi": 6,
    "generation-vii": 7, "generation-viii": 8, "generation-ix": 9,
}

def obtener_generaciones():
    """Devuelve un dict {nombre_pokemon: numero_generacion}."""
    nombre_a_gen = {}
    for gen_url_name, gen_num in ROMAN_TO_GEN.items():
        resp = requests.get(f"{BASE_URL}/generation/{gen_num}", timeout=30)
        resp.raise_for_status()
        data = resp.json()
        for especie in data["pokemon_species"]:
            nombre_a_gen[especie["name"]] = gen_num
    return nombre_a_gen

nombre_a_generacion = obtener_generaciones()
print(f"Especies mapeadas a una generación: {len(nombre_a_generacion)}")

Especies mapeadas a una generación: 1025


Se define el conjunto `LEGENDARIOS_MITICOS`, con los nombres (tal como aparecen en la PokeAPI) de todos los Pokémon legendarios y míticos de las generaciones 1 a 9. Se usará más adelante para marcar la columna `Legendary` de cada registro.

In [3]:
# Lista de Pokémon legendarios y míticos (gen 1-9). Se usa por nombre en minúsculas
# tal como aparece en la PokeAPI. Puedes ampliarla si la PokeAPI agrega Pokémon nuevos.
LEGENDARIOS_MITICOS = {
    # Gen 1
    "articuno", "zapdos", "moltres", "mewtwo", "mew",
    # Gen 2
    "raikou", "entei", "suicune", "lugia", "ho-oh", "celebi",
    # Gen 3
    "regirock", "regice", "registeel", "latias", "latios", "kyogre",
    "groudon", "rayquaza", "jirachi", "deoxys",
    # Gen 4
    "uxie", "mesprit", "azelf", "dialga", "palkia", "heatran", "regigigas",
    "giratina", "cresselia", "phione", "manaphy", "darkrai", "shaymin", "arceus",
    # Gen 5
    "victini", "cobalion", "terrakion", "virizion", "tornadus", "thundurus",
    "reshiram", "zekrom", "landorus", "kyurem", "keldeo", "meloetta", "genesect",
    # Gen 6
    "xerneas", "yveltal", "zygarde", "diancie", "hoopa", "volcanion",
    # Gen 7
    "type-null", "silvally", "tapu-koko", "tapu-lele", "tapu-bulu", "tapu-fini",
    "cosmog", "cosmoem", "solgaleo", "lunala", "necrozma", "magearna", "marshadow",
    "zeraora", "meltan", "melmetal",
    # Gen 8
    "zacian", "zamazenta", "eternatus", "kubfu", "urshifu", "zarude",
    "regieleki", "regidrago", "glastrier", "spectrier", "calyrex",
    # Gen 9
    "wo-chien", "chien-pao", "ting-lu", "chi-yu", "koraidon", "miraidon",
    "walking-wake", "iron-leaves", "okidogi", "munkidori", "fezandipiti",
    "ogerpon", "terapagos", "pecharunt",
}

Se actualiza la librería `ipywidgets`, necesaria para que los controles interactivos del notebook funcionen correctamente.

In [4]:
!pip install --upgrade ipywidgets

Se define la función `obtener_pokemon`, que consulta el endpoint `/pokemon/{id}` para obtener nombre, tipos, estadísticas base y sprite de un Pokémon, y arma el diccionario con las columnas que se usarán en el dataset. Después se descargan en paralelo (con `ThreadPoolExecutor`) los datos de todos los Pokémon (1 a 1025) para acelerar el proceso.

In [ ]:
def obtener_pokemon(pid):
    """Consulta /pokemon/{id} y devuelve un dict con los campos que nos interesan."""
    resp = requests.get(f"{BASE_URL}/pokemon/{pid}", timeout=30)
    if resp.status_code != 200:
        return None
    data = resp.json()

    nombre = data["name"]
    tipos = [t["type"]["name"] for t in sorted(data["types"], key=lambda t: t["slot"])]
    tipo1 = tipos[0].capitalize() if len(tipos) > 0 else None
    tipo2 = tipos[1].capitalize() if len(tipos) > 1 else None

    stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}

    sprite_url = data["sprites"]["front_default"]
    if not sprite_url:
        # Algunos Pokémon no tienen sprite clásico; usamos el artwork oficial como respaldo
        sprite_url = data["sprites"]["other"]["official-artwork"]["front_default"]

    generacion = nombre_a_generacion.get(nombre)

    return {
        "ID": data["id"],
        "Name": nombre.replace("-", " ").title(),
        "Generation": generacion,
        "Type 1": tipo1,
        "Type 2": tipo2,
        "HP": stats.get("hp", 0),
        "Attack": stats.get("attack", 0),
        "Defense": stats.get("defense", 0),
        "Sp. Atk": stats.get("special-attack", 0),
        "Sp. Def": stats.get("special-defense", 0),
        "Speed": stats.get("speed", 0),
        "Legendary": nombre in LEGENDARIOS_MITICOS,
        "Sprite URL": sprite_url,
    }

registros = []
errores = []

with ThreadPoolExecutor(max_workers=20) as executor:
    futuros = {executor.submit(obtener_pokemon, pid): pid for pid in range(1, N_POKEMON + 1)}
    for futuro in tqdm(as_completed(futuros), total=len(futuros), desc="Descargando Pokémon"):
        pid = futuros[futuro]
        try:
            resultado = futuro.result()
            if resultado is not None:
                registros.append(resultado)
            else:
                errores.append(pid)
        except Exception as e:
            errores.append(pid)

print(f"Pokémon descargados correctamente: {len(registros)}")
print(f"IDs con error (se ignoran): {len(errores)}")

Descargando Pokémon:  90%|████████▉ | 921/1025 [01:53<00:09, 11.14it/s]

## 3. Selección y justificación de las variables estadísticas

Antes de construir el DataFrame final, se eligen las variables numéricas que se usarán para caracterizar a cada Pokémon:

- **HP, Attack, Defense, Sp. Atk, Sp. Def, Speed**: son las **seis estadísticas base** que la PokeAPI expone para todo Pokémon (endpoint `/pokemon/{id}`), sin importar la generación. Se eligen estas seis y no otras (como altura, peso o experiencia base) porque son las que definen directamente el desempeño en combate y son las que tradicionalmente se usan en los datasets de Pokémon para comparar especies entre sí.
- No se incluyen columnas como *height* o *weight* porque no aportan información de rendimiento competitivo, que es el foco del análisis (comparar Pokémon por generación y tipo según qué tan "fuertes" son en promedio).
- A partir de estas seis variables se calculará una sola métrica resumen, `promedio_estadisticas`, que se usará como eje Z del gráfico 3D — de ahí que sea importante justificar que las seis tienen la misma escala aproximada (0-255) y por lo tanto pueden promediarse directamente sin normalizar.

## 4. Construcción del DataFrame y cálculo del promedio de estadísticas

$$\text{promedio\_estadisticas} = \dfrac{HP + Attack + Defense + Sp.\ Atk + Sp.\ Def + Speed}{6}$$

Se construye el DataFrame a partir de los registros descargados, se calcula la columna `promedio_estadisticas` (media de las seis estadísticas base), se descartan los Pokémon sin generación asignada y se guarda el resultado en un archivo CSV local para no tener que repetir la descarga en el futuro.

In [ ]:
df = pd.DataFrame(registros).sort_values("ID").reset_index(drop=True)

stat_cols = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
df["promedio_estadisticas"] = df[stat_cols].mean(axis=1).round(2)

# Descartamos filas sin generación asignada (formas especiales no listadas en /generation)
df = df.dropna(subset=["Generation", "Type 1"]).copy()
df["Generation"] = df["Generation"].astype(int)

df.to_csv(CSV_PATH, index=False, encoding="utf-8")
print(f"Dataset guardado en: {CSV_PATH}")
df.head(10)

### 5. Descripción del contenido del dataset

El dataset construido (`pokemon_dataset.csv`) contiene un registro por cada Pokémon (generaciones 1 a 9) con las siguientes columnas:

| Columna | Descripción |
|---|---|
| `ID` | Número de Pokédex nacional |
| `Name` | Nombre del Pokémon |
| `Generation` | Generación a la que pertenece (1-9) |
| `Type 1` | Tipo principal |
| `Type 2` | Tipo secundario (puede ser nulo si el Pokémon es de un solo tipo) |
| `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, `Speed` | Las seis estadísticas base |
| `Legendary` | `True`/`False`, si es legendario o mítico |
| `Sprite URL` | URL de la imagen del Pokémon en la PokeAPI |
| `promedio_estadisticas` | Promedio de las seis estadísticas base |

**Origen:** los datos provienen de la [PokeAPI](https://pokeapi.co/), una API REST pública y gratuita con información de todos los juegos de la franquicia Pokémon.

### 6. Limpieza y normalización de columnas y valores categóricos

Se revisa que los nombres de las columnas sean consistentes y que los valores de las columnas categóricas (`Type 1`, `Type 2`) estén normalizados (sin espacios extra y con el mismo formato de capitalización), ya que la PokeAPI a veces entrega los nombres de tipos en minúsculas.

In [ ]:
print("--- ANTES de la limpieza ---")
print("Columnas:", list(df.columns))
print("Ejemplos de Type 1 sin normalizar:", df["Type 1"].str.strip().str.lower().unique()[:5])

# Normalizamos nombres de columnas: sin espacios extremos y formato consistente
df.columns = [c.strip() for c in df.columns]

# Normalizamos texto de las columnas categóricas: quitamos espacios y usamos Capitalized Case
for col in ["Type 1", "Type 2", "Name"]:
    df[col] = df[col].astype(str).str.strip()
    if col != "Name":
        df[col] = df[col].str.capitalize()
    df.loc[df[col].isin(["Nan", "None", "nan"]), col] = np.nan

print("\n--- DESPUÉS de la limpieza ---")
print("Columnas:", list(df.columns))
print("Valores únicos de Type 1:", sorted(df["Type 1"].dropna().unique()))

### 7. Identificación y tratamiento de valores nulos y registros duplicados

Se cuentan los valores nulos por columna y los registros duplicados **antes** de tratarlos, se aplica la limpieza correspondiente, y se vuelve a contar **después** para verificar el resultado.

In [ ]:
print("--- ANTES de tratar nulos y duplicados ---")
print("Valores nulos por columna:")
print(df.isnull().sum())
print("\nRegistros duplicados (por ID):", df.duplicated(subset=["ID"]).sum())
print("Filas totales:", df.shape[0])

# Type 2 puede ser legítimamente nulo (Pokémon de un solo tipo), no se elimina por eso.
# Eliminamos duplicados exactos por ID, si los hubiera, y filas sin Type 1 (dato crítico para el análisis)
df = df.drop_duplicates(subset=["ID"]).copy()
df = df.dropna(subset=["Type 1"]).copy()

print("\n--- DESPUÉS de tratar nulos y duplicados ---")
print("Valores nulos por columna:")
print(df.isnull().sum())
print("\nRegistros duplicados (por ID):", df.duplicated(subset=["ID"]).sum())
print("Filas totales:", df.shape[0])

### (Opcional) Cargar el dataset ya guardado

Si ya ejecutaste las celdas de descarga una vez, puedes saltarte la sección 2 en futuras ejecuciones y simplemente cargar el CSV guardado:

Celda opcional (comentada): si ya se generó el CSV anteriormente, aquí se puede cargar directamente desde disco en lugar de repetir toda la descarga de la sección 2.

In [ ]:
# df = pd.read_csv(CSV_PATH)
# df.head()

## 8. Inspección inicial del dataset

Inspección inicial del dataset ya construido: primeras filas (`head`), dimensiones (`shape`), tipos de datos y nulos por columna (`info`), y estadísticos generales (`describe`). Después se revisa el número de Pokémon por generación, por tipo principal, y el promedio de estadísticas por generación.

In [ ]:
print("--- head() ---")
display(df.head())

print("\n--- shape ---")
print(df.shape)

print("\n--- info() ---")
df.info()

print("\n--- describe() ---")
display(df.describe())

In [ ]:
print("Dimensiones:", df.shape)
print("\nPokémon por generación:")
print(df["Generation"].value_counts().sort_index())
print("\nPokémon por tipo principal:")
print(df["Type 1"].value_counts())
print("\npromedio_estadisticas por generación:")
print(df.groupby("Generation")["promedio_estadisticas"].mean().round(2))

### 9. Análisis estadístico descriptivo completo

Se calculan explícitamente la **media**, **mediana**, **mínimo**, **máximo** y **desviación estándar** de las seis estadísticas base y del `promedio_estadisticas`, tanto de forma global como agrupadas por generación.

In [ ]:
cols_analisis = stat_cols + ["promedio_estadisticas"]

resumen = pd.DataFrame({
    "media": df[cols_analisis].mean().round(2),
    "mediana": df[cols_analisis].median().round(2),
    "minimo": df[cols_analisis].min(),
    "maximo": df[cols_analisis].max(),
    "desviacion_estandar": df[cols_analisis].std().round(2),
})
print("--- Análisis estadístico descriptivo global ---")
display(resumen)

print("\n--- promedio_estadisticas por generación (media, mediana, min, max, std) ---")
display(df.groupby("Generation")["promedio_estadisticas"].agg(
    media="mean", mediana="median", minimo="min", maximo="max", desviacion_estandar="std"
).round(2))

## 10. Descarga de sprites para graficar

Graficar los ~1000 sprites al mismo tiempo en 3D generaría un gráfico ilegible y muy lento de renderizar. Por eso se selecciona una **muestra representativa** (configurable con `MUESTRA_POR_GENERACION`) y solo se descargan las imágenes de esa muestra.

Cambia `MUESTRA_POR_GENERACION` a un número mayor (o a `None` para usar el dataset completo) si quieres más puntos — ten en cuenta que el gráfico será más pesado.

Se selecciona la muestra de Pokémon que se va a graficar en 3D. Si `MUESTRA_POR_GENERACION` es `None` se usa el dataset completo; si se le asigna un número, se toma una muestra aleatoria de ese tamaño por cada generación, para mantener el gráfico legible y ligero.

In [ ]:
MUESTRA_POR_GENERACION = None   # None = usar TODOS los Pokémon del dataset

if MUESTRA_POR_GENERACION is None:
    df_plot = df.copy()
else:
    partes = []
    for gen, grupo in df.groupby("Generation"):
        n = min(len(grupo), MUESTRA_POR_GENERACION)
        partes.append(grupo.sample(n, random_state=SEMILLA))
    df_plot = pd.concat(partes).reset_index(drop=True)

print(f"Pokémon seleccionados para el gráfico 3D: {len(df_plot)}")

Se descargan (o se reutilizan si ya existen en disco) los sprites de los Pokémon seleccionados en `df_plot`, en paralelo con `ThreadPoolExecutor` (8 workers, para no saturar la PokeAPI y provocar más fallos por límite de solicitudes). Cada descarga se reintenta hasta `MAX_INTENTOS` veces con una pequeña espera creciente entre intentos (backoff), y si aun así falla, se registra el Pokémon y el motivo del error en `errores_sprites` (guardado también en `errores_sprites.csv`) para poder diagnosticarlo o reintentarlo después. La ruta local de cada imagen descargada se guarda en la columna `Sprite Path`.

In [ ]:
MAX_INTENTOS = 3        # numero de reintentos por sprite antes de darlo por perdido
ESPERA_BASE = 1.5       # segundos; se multiplica por el numero de intento (backoff simple)

def descargar_sprite(row):
    """Descarga el sprite de una fila del dataset, con reintentos y registro del motivo de falla.

    Devuelve una tupla (ruta_o_None, error_o_None) para poder distinguir
    exitos de fallos y saber por que fallo cada uno.
    """
    ruta = os.path.join(SPRITE_DIR, f"{row.ID}.png")
    if os.path.exists(ruta):
        return ruta, None

    ultimo_error = None
    for intento in range(1, MAX_INTENTOS + 1):
        try:
            resp = requests.get(row["Sprite URL"], timeout=30)
            resp.raise_for_status()
            with open(ruta, "wb") as f:
                f.write(resp.content)
            return ruta, None
        except Exception as e:
            ultimo_error = f"{type(e).__name__}: {e}"
            if intento < MAX_INTENTOS:
                time.sleep(ESPERA_BASE * intento)  # backoff: espera un poco mas en cada intento

    return None, ultimo_error

rutas_sprites = []
errores_sprites = []  # lista de (ID, Name, error) para diagnosticar que y por que fallo

with ThreadPoolExecutor(max_workers=8) as executor:
    filas = [r for _, r in df_plot.iterrows()]
    resultados = list(tqdm(
        executor.map(descargar_sprite, filas),
        total=len(filas), desc="Descargando sprites"
    ))

for fila, (ruta, error) in zip(filas, resultados):
    rutas_sprites.append(ruta)
    if error is not None:
        errores_sprites.append((fila["ID"], fila["Name"], error))

df_plot = df_plot.copy()
df_plot["Sprite Path"] = rutas_sprites
df_plot = df_plot.dropna(subset=["Sprite Path"]).reset_index(drop=True)

print(f"Sprites descargados correctamente: {len(df_plot)}")
print(f"Sprites que fallaron tras {MAX_INTENTOS} intentos: {len(errores_sprites)}")

if errores_sprites:
    df_errores_sprites = pd.DataFrame(errores_sprites, columns=["ID", "Name", "Error"])
    print("\nDetalle de los primeros errores:")
    print(df_errores_sprites.head(20))
    # Se guarda el detalle completo por si se quiere reintentar solo esos IDs mas tarde
    df_errores_sprites.to_csv(os.path.join(DATA_DIR, "errores_sprites.csv"), index=False)


## 11. Scatter 3D Plot con sprites (Matplotlib)

Matplotlib no soporta imágenes como símbolos de forma nativa en un `scatter3D`. El truco consiste en:

1. Proyectar cada punto 3D a coordenadas 2D de pantalla con `proj3d.proj_transform` según el ángulo de cámara actual (`ax.get_proj()`).
2. Colocar el sprite en esa posición 2D usando `OffsetImage` + `AnnotationBbox`.
3. Volver a proyectar y reposicionar los sprites cada vez que el usuario rota el gráfico (evento `button_release_event`), para que las imágenes 'seudo-floten' en 3D.

> Para que la rotación interactiva funcione dentro de Jupyter necesitas un backend interactivo, por ejemplo ejecutando `%matplotlib widget` (requiere `ipympl`) en una celda antes de graficar. Con el backend `inline` por defecto verás una vista estática.

Se define la paleta de colores oficiales por tipo de Pokémon, se codifica cada tipo principal como un número (para poder ubicarlo en el eje Y) y se arman los arreglos `xs`, `ys`, `zs`, `colores` y `rutas` que alimentarán el gráfico 3D con Matplotlib.

In [ ]:
# Colores oficiales aproximados por tipo de Pokémon (en formato hex, válido para Matplotlib y Plotly)
COLOR_TIPO_HEX = {
    "Normal": "#A8A878", "Fire": "#F08030", "Water": "#6890F0", "Electric": "#F8D030",
    "Grass": "#78C850", "Ice": "#98D8D8", "Fighting": "#C03028", "Poison": "#A040A0",
    "Ground": "#E0C068", "Flying": "#A890F0", "Psychic": "#F85888", "Bug": "#A8B820",
    "Rock": "#B8A038", "Ghost": "#705898", "Dragon": "#7038F8", "Dark": "#705848",
    "Steel": "#B8B8D0", "Fairy": "#EE99AC",
}

tipos_unicos = sorted(df_plot["Type 1"].unique())
tipo_a_codigo = {tipo: i for i, tipo in enumerate(tipos_unicos)}

# Si algún tipo no está en el diccionario (por mayúsculas/minúsculas u otro motivo), le asignamos gris
color_por_tipo = {tipo: COLOR_TIPO_HEX.get(tipo, "#CCCCCC") for tipo in tipos_unicos}

# Codificación numérica del eje Y (Tipo principal)
tipo_a_codigo = {tipo: i for i, tipo in enumerate(tipos_unicos)}

xs = df_plot["Generation"].to_numpy(dtype=float)
ys = df_plot["Type 1"].map(tipo_a_codigo).to_numpy(dtype=float)
zs = df_plot["promedio_estadisticas"].to_numpy(dtype=float)
colores = [color_por_tipo[t] for t in df_plot["Type 1"]]
rutas = df_plot["Sprite Path"].tolist()

Se define y ejecuta la función `graficar_scatter3d_sprites`, que dibuja el scatter 3D con Matplotlib y coloca el sprite de cada Pokémon sobre su punto correspondiente, reproyectando y reposicionando las imágenes cada vez que se rota el gráfico para simular que 'flotan' en el espacio 3D.

In [ ]:
def graficar_scatter3d_sprites(xs, ys, zs, rutas, colores, zoom=0.15, figsize=(22, 16)):
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection="3d")

    # Puntos de color como referencia visual bajo cada sprite
    ax.scatter(xs, ys, zs, c=colores, s=15, depthshade=True, alpha=0.7)

    imagenes = [plt.imread(r) for r in rutas]
    cajas = []
    for x, y, z, img in zip(xs, ys, zs, imagenes):
        x2, y2, _ = proj3d.proj_transform(x, y, z, ax.get_proj())
        caja = AnnotationBbox(OffsetImage(img, zoom=zoom), (x2, y2), frameon=False, xycoords="data", pad=0)
        cajas.append(ax.add_artist(caja))

    def actualizar_posiciones(event=None):
        for caja, x, y, z in zip(cajas, xs, ys, zs):
            x2, y2, _ = proj3d.proj_transform(x, y, z, ax.get_proj())
            caja.xybox = (x2, y2)
            caja.xy = (x2, y2)
        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_release_event", actualizar_posiciones)

    ax.set_xlabel("Generación", fontsize=12, labelpad=12)
    ax.set_ylabel("Tipo principal", fontsize=12, labelpad=60)
    ax.set_zlabel("Promedio de estadísticas", fontsize=12, labelpad=10)

    ax.set_xticks(sorted(df_plot["Generation"].unique()))
    ax.set_yticks(list(tipo_a_codigo.values()))
    ax.set_yticklabels(list(tipo_a_codigo.keys()), fontsize=8)

    ax.set_title("Pokémon por Generación, Tipo y Promedio de Estadísticas (sprites)", fontsize=14)

    leyenda = [plt.Line2D([0], [0], marker="o", color="w", label=t,
                          markerfacecolor=color_por_tipo[t], markersize=8) for t in tipos_unicos]
    ax.legend(handles=leyenda, bbox_to_anchor=(1.1, 1), loc="upper left",
              fontsize=8, title="Tipo principal", ncol=1)

    plt.tight_layout()
    return fig, ax

fig, ax = graficar_scatter3d_sprites(xs, ys, zs, rutas, colores, zoom=0.15)
plt.show()

## 12. Versión interactiva con Plotly (rotación y zoom fluidos)

Plotly no permite usar imágenes como símbolo de un `Scatter3d`, pero sí ofrece una exploración 3D totalmente interactiva (rotar, hacer zoom, filtrar por leyenda) y tooltips con toda la información de cada Pokémon al pasar el mouse — una buena visualización complementaria a la de sprites.

Se construye una versión interactiva con Plotly y Dash: se codifican los sprites en base64, se arma un `Scatter3d` por cada tipo principal (para poder filtrar por leyenda) con tooltips, y se define una app de Dash que muestra el sprite y las estadísticas del Pokémon al hacer clic sobre un punto del gráfico.

In [ ]:
#pip install dash  (si no lo tienes instalado, descomenta la línea de abajo)
!pip install dash

In [ ]:
from dash import Dash, dcc, html, Input, Output, no_update
import plotly.graph_objects as go
import base64


# ============================================================
# CARGAR SPRITES EN MEMORIA
# ============================================================

sprite_base64 = {}

for _, row in df_plot.iterrows():
    with open(row["Sprite Path"], "rb") as f:
        codificado = base64.b64encode(f.read()).decode()

    sprite_base64[int(row["ID"])] = (
        f"data:image/png;base64,{codificado}"
    )

print("Sprites cargados en memoria:", len(sprite_base64))
print("Ejemplo de claves:", list(sprite_base64.keys())[:5])


# ============================================================
# CREAR FIGURA DE PLOTLY
# ============================================================

fig_plotly = go.Figure()

for tipo in tipos_unicos:

    subset = df_plot[
        df_plot["Type 1"] == tipo
    ]

    # customdata incluye:
    # ID, nombre, estadísticas, tipo,
    # generación y promedio
    customdata_array = subset[
        [
            "ID",
            "Name",
            "HP",
            "Attack",
            "Defense",
            "Sp. Atk",
            "Sp. Def",
            "Speed",
            "Type 1",
            "Generation",
            "promedio_estadisticas"
        ]
    ].to_numpy()

    fig_plotly.add_scatter3d(

        x=subset["Generation"],

        y=subset["Type 1"].map(
            tipo_a_codigo
        ),

        z=subset[
            "promedio_estadisticas"
        ],

        mode="markers",

        name=tipo,

        marker=dict(
            size=5,
            color=color_por_tipo[tipo],
            line=dict(
                width=0.5,
                color="DarkSlateGrey"
            )
        ),

        customdata=customdata_array,

        hovertemplate=(
            "<b>%{customdata[1]}</b><br>"
            "Tipo: %{customdata[8]}<br>"
            "Generación: %{customdata[9]}<br>"
            "Promedio de estadísticas: %{customdata[10]}"
            "<extra></extra>"
        ),
    )


# ============================================================
# CONFIGURACIÓN DEL GRÁFICO
# ============================================================

fig_plotly.update_layout(

    # Gráfica más pequeña
    height=550,

    title=(
        "Pokémon: Generación x Tipo principal "
        "x Promedio de estadísticas"
    ),

    scene=dict(

        xaxis_title="Generación",

        yaxis_title="Tipo principal",

        zaxis_title="Promedio de estadísticas",

        yaxis=dict(
            tickmode="array",

            tickvals=list(
                tipo_a_codigo.values()
            ),

            ticktext=list(
                tipo_a_codigo.keys()
            ),
        ),
    ),
)


# ============================================================
# CREAR APLICACIÓN DASH
# ============================================================

app = Dash(__name__)


# ============================================================
# DISEÑO DE LA APLICACIÓN
# ============================================================

app.layout = html.Div(

    [

        # ----------------------------------------------------
        # GRÁFICA
        # ----------------------------------------------------

        html.Div(

            dcc.Graph(

                id="grafico-3d",

                figure=fig_plotly,

                # Gráfica más compacta
                style={
                    "height": "550px"
                },
            ),

            style={
                "flex": "1 1 auto",
                "minWidth": "0",
            },
        ),


        # ----------------------------------------------------
        # PANEL DEL POKÉMON
        # ----------------------------------------------------

        html.Div(

            id="panel-sprite",

            children=html.P(
                "Haz clic en un punto del gráfico "
                "para ver el Pokémon."
            ),

            style={

                "flex": "0 0 220px",

                "textAlign": "center",

                "padding": "20px",

                "border": "1px solid #ccc",

                "borderRadius": "10px",

                "marginLeft": "16px",

                "alignSelf": "flex-start",
            },
        ),
    ],


    # --------------------------------------------------------
    # CONTENEDOR PRINCIPAL
    # --------------------------------------------------------

    style={

        "display": "flex",

        "flexWrap": "nowrap",

        "alignItems": "flex-start",

        # Altura mínima de la aplicación
        "minHeight": "600px",

        # Permitir desplazamiento vertical
        "overflowY": "auto",

        # Evitar desplazamiento horizontal
        "overflowX": "hidden",
    },
)


# ============================================================
# CALLBACK
# ============================================================

@app.callback(

    Output(
        "panel-sprite",
        "children"
    ),

    Input(
        "grafico-3d",
        "clickData"
    ),
)


def mostrar_sprite(clickData):

    # --------------------------------------------------------
    # SIN SELECCIÓN
    # --------------------------------------------------------

    if clickData is None:

        return html.P(
            "Haz clic en un punto del gráfico "
            "para ver el Pokémon."
        )


    # --------------------------------------------------------
    # OBTENER PUNTO SELECCIONADO
    # --------------------------------------------------------

    punto = clickData["points"][0]


    # --------------------------------------------------------
    # COMPROBAR CUSTOMDATA
    # --------------------------------------------------------

    if "customdata" not in punto:

        return html.P(
            "El punto no trae customdata "
            "(revisa la consola)."
        )


    # --------------------------------------------------------
    # EXTRAER DATOS
    # --------------------------------------------------------

    (
        pid,
        nombre,
        hp,
        ataque,
        defensa,
        sp_atk,
        sp_def,
        speed,
        tipo,
        gen,
        promedio
    ) = punto["customdata"]


    # Convertir ID a entero
    pid = int(float(pid))


    # --------------------------------------------------------
    # BUSCAR SPRITE
    # --------------------------------------------------------

    imagen = sprite_base64.get(
        pid,
        ""
    )


    # --------------------------------------------------------
    # SI NO EXISTE EL SPRITE
    # --------------------------------------------------------

    if not imagen:

        return html.P(
            f"No se encontró sprite para el ID "
            f"{pid} (nombre: {nombre})."
        )


    # --------------------------------------------------------
    # MOSTRAR INFORMACIÓN
    # --------------------------------------------------------

    return html.Div(

        [

            # Sprite
            html.Img(

                src=imagen,

                style={
                    "width": "96px",
                    "imageRendering": "pixelated"
                },
            ),


            # Nombre
            html.H4(

                nombre,

                style={
                    "margin": "10px 0 5px 0"
                },
            ),


            # Tipo
            html.P(
                f"Tipo: {tipo}"
            ),


            # Generación
            html.P(
                f"Generación: {gen}"
            ),


            # Estadísticas
            html.P(
                f"HP: {hp}"
            ),

            html.P(
                f"Attack: {ataque}"
            ),

            html.P(
                f"Defense: {defensa}"
            ),

            html.P(
                f"Sp. Atk: {sp_atk}"
            ),

            html.P(
                f"Sp. Def: {sp_def}"
            ),

            html.P(
                f"Speed: {speed}"
            ),


            # Promedio
            html.P(
                f"Promedio: {promedio}"
            ),
        ]
    )


# ============================================================
# EJECUTAR DASH
# ============================================================

app.run(
    jupyter_mode="inline",
    port=8055
)

## 13. Filtros interactivos

Se agregan controles interactivos con `ipywidgets` para filtrar el gráfico 3D por **generación**, **tipo principal** y **rango de `promedio_estadisticas`**. Cada vez que se cambia un control, el gráfico se reconstruye solo con los Pokémon que cumplen el filtro.

In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display

gen_opciones = ["Todas"] + sorted(df_plot["Generation"].unique().tolist())
tipo_opciones = ["Todos"] + tipos_unicos

filtro_generacion = widgets.Dropdown(options=gen_opciones, value="Todas", description="Generación:")
filtro_tipo = widgets.Dropdown(options=tipo_opciones, value="Todos", description="Tipo:")
filtro_rango = widgets.IntRangeSlider(
    value=[int(df_plot["promedio_estadisticas"].min()), int(df_plot["promedio_estadisticas"].max())],
    min=int(df_plot["promedio_estadisticas"].min()), max=int(df_plot["promedio_estadisticas"].max()),
    step=1, description="Promedio:", continuous_update=False,
)

salida_filtro = widgets.Output()

def actualizar_grafico_filtrado(generacion, tipo, rango):
    datos = df_plot.copy()
    if generacion != "Todas":
        datos = datos[datos["Generation"] == generacion]
    if tipo != "Todos":
        datos = datos[datos["Type 1"] == tipo]
    datos = datos[datos["promedio_estadisticas"].between(rango[0], rango[1])]

    fig_filtrada = go.Figure()
    for t in sorted(datos["Type 1"].unique()):
        subset = datos[datos["Type 1"] == t]
        fig_filtrada.add_scatter3d(
            x=subset["Generation"], y=subset["Type 1"].map(tipo_a_codigo), z=subset["promedio_estadisticas"],
            mode="markers", name=t,
            marker=dict(size=5, color=color_por_tipo.get(t, "#CCCCCC")),
            hovertext=subset["Name"], hoverinfo="text",
        )
    fig_filtrada.update_layout(
        height=650,
        title=f"Filtro: Generación={generacion} | Tipo={tipo} | Promedio={rango[0]}-{rango[1]} ({len(datos)} Pokémon)",
        scene=dict(xaxis_title="Generación", yaxis_title="Tipo principal", zaxis_title="Promedio de estadísticas"),
    )

    with salida_filtro:
        salida_filtro.clear_output(wait=True)
        fig_filtrada.show()

def _on_change(change):
    actualizar_grafico_filtrado(filtro_generacion.value, filtro_tipo.value, filtro_rango.value)

filtro_generacion.observe(_on_change, names="value")
filtro_tipo.observe(_on_change, names="value")
filtro_rango.observe(_on_change, names="value")

ipy_display(widgets.HBox([filtro_generacion, filtro_tipo, filtro_rango]))
ipy_display(salida_filtro)
actualizar_grafico_filtrado(filtro_generacion.value, filtro_tipo.value, filtro_rango.value)

## 14. Exportar la visualización interactiva a HTML

Se exporta el gráfico 3D interactivo de Plotly a un archivo `.html` independiente, que conserva la rotación, el zoom y los tooltips al abrirse en cualquier navegador, sin necesidad de tener Python instalado.

In [ ]:
ruta_html = os.path.join(DATA_DIR, "pokemon_scatter3d.html")
fig_plotly.write_html(ruta_html, include_plotlyjs="cdn", full_html=True)
print(f"Visualización interactiva exportada en: {ruta_html}")

## 15. Agregar Pokémon fan-made a la gráfica 3D

Antes de correr la siguiente celda, coloca la carpeta `pokemon_data_custom` (con los 8 sprites: `lumibrote.png`, `floralis.png`, `floralume.png`, `mega_floralume.png`, `voltraptor.png`, `ferragon.png`, `abyssphinx.png`, `chronarion.png`) en la misma carpeta que este notebook. Esta celda reutiliza el `df_plot` generado arriba, así que corre todo el notebook normal primero.

In [ ]:
CUSTOM_SPRITE_DIR = os.path.join(DATA_DIR, "sprites")

datos_custom = [
    {
        "ID": 9001,
        "Name": "Auricornia",
        "Type 1": "Artificial",
        "Type 2": "Luminárico",
        "HP": 90,
        "Attack": 75,
        "Defense": 86,
        "Sp. Atk": 112,
        "Sp. Def": 108,
        "Speed": 89,
        "Sprite": "auricornia.png"
    },

    {
        "ID": 9002,
        "Name": "Estrelux",
        "Type 1": "Artificial",
        "Type 2": "Estelar",
        "HP": 70,
        "Attack": 60,
        "Defense": 72,
        "Sp. Atk": 125,
        "Sp. Def": 100,
        "Speed": 123,
        "Sprite": "estrelux.png"
    },

    {
        "ID": 9003,
        "Name": "Ferrolobo",
        "Type 1": "Artificial",
        "Type 2": "Acero",
        "HP": 82,
        "Attack": 110,
        "Defense": 92,
        "Sp. Atk": 68,
        "Sp. Def": 84,
        "Speed": 104,
        "Sprite": "ferrolobo.png"
    },

    {
        "ID": 9004,
        "Name": "Draciria",
        "Type 1": "Artificial",
        "Type 2": "Fuego",
        "HP": 95,
        "Attack": 88,
        "Defense": 90,
        "Sp. Atk": 118,
        "Sp. Def": 99,
        "Speed": 70,
        "Sprite": "draciria.png"
    },

    {
        "ID": 9005,
        "Name": "Mecanursa",
        "Type 1": "Artificial",
        "Type 2": "Hielo",
        "HP": 110,
        "Attack": 120,
        "Defense": 118,
        "Sp. Atk": 65,
        "Sp. Def": 90,
        "Speed": 47,
        "Sprite": "mecanursa.png"
    }
]


# ============================================================
# CREAR DATAFRAME
# ============================================================

df_custom = pd.DataFrame(datos_custom)

# Todas las criaturas personalizadas pertenecen a la generación 9
df_custom["Generation"] = 9


# ============================================================
# ESTADÍSTICAS
# ============================================================

# Aseguramos que stat_cols tenga las estadísticas correctas
stat_cols = [
    "HP",
    "Attack",
    "Defense",
    "Sp. Atk",
    "Sp. Def",
    "Speed"
]

# Promedio de estadísticas
df_custom["promedio_estadisticas"] = (
    df_custom[stat_cols]
    .mean(axis=1)
    .round(2)
)


# ============================================================
# SPRITES
# ============================================================

# Crear ruta completa del sprite
df_custom["Sprite Path"] = df_custom["Sprite"].apply(
    lambda n: os.path.join(CUSTOM_SPRITE_DIR, n)
)

# Ya no necesitamos la columna Sprite
df_custom = df_custom.drop(columns=["Sprite"])


# ============================================================
# COMPROBAR SPRITES
# ============================================================

faltantes = df_custom[
    ~df_custom["Sprite Path"].apply(os.path.exists)
]

if len(faltantes) > 0:
    print("⚠️ No se encontraron estos sprites en disco:")
    print()

    for _, pokemon in faltantes.iterrows():
        print(
            f"   ❌ {pokemon['Name']} -> "
            f"{pokemon['Sprite Path']}"
        )

else:
    print("✅ Todos los sprites personalizados fueron encontrados.")


# ============================================================
# AGREGAR AL DATAFRAME PRINCIPAL
# ============================================================

df_plot = pd.concat(
    [df_plot, df_custom],
    ignore_index=True
)


# ============================================================
# COLORES DE TIPOS
# ============================================================

COLOR_TIPO_HEX.update({
    "Artificial": "#22A6D5",
    "Luminárico": "#B47CFF",
    "Estelar": "#D99A20",
    "Acero": "#718096",
    "Fuego": "#F0445D",
    "Hielo": "#38BDF8",

    # Tipos que ya tenías
    "Electránico": "#c9a3ff",
    "Ferromagnético": "#7e57c2",
    "Enigmático": "#3949ab",
    "Cronal": "#9575cd",
})


# ============================================================
# TIPOS PARA EL GRÁFICO
# ============================================================

# El eje principal utiliza Type 1
tipos_unicos = sorted(
    df_plot["Type 1"]
    .dropna()
    .unique()
)

tipo_a_codigo = {
    tipo: i
    for i, tipo in enumerate(tipos_unicos)
}

color_por_tipo = {
    tipo: COLOR_TIPO_HEX.get(
        tipo,
        "#CCCCCC"
    )
    for tipo in tipos_unicos
}


# ============================================================
# DATOS PARA EL GRÁFICO 3D
# ============================================================

xs = df_plot["Generation"].to_numpy(
    dtype=float
)

ys = df_plot["Type 1"].map(
    tipo_a_codigo
).to_numpy(
    dtype=float
)

zs = df_plot[
    "promedio_estadisticas"
].to_numpy(
    dtype=float
)

colores = [
    color_por_tipo.get(
        tipo,
        "#CCCCCC"
    )
    for tipo in df_plot["Type 1"]
]

rutas = df_plot[
    "Sprite Path"
].tolist()


# ============================================================
# INFORMACIÓN FINAL
# ============================================================

print()
print(
    "✅ Total de Pokémon en el gráfico "
    "(incluye fan-made):",
    len(df_plot)
)

print()
print("✅ Pokémon personalizados agregados:")

for pokemon in datos_custom:
    print(
        f"   {pokemon['ID']} - "
        f"{pokemon['Name']} "
        f"({pokemon['Type 1']} / "
        f"{pokemon['Type 2']})"
    )

In [ ]:
import plotly.graph_objects as go
from functools import lru_cache

PIXEL_SPRITE = 32   # 32 

offsets = [
    (0.00, 0.00), (-0.23, -0.23), (0.23, 0.23), (-0.23, 0.23), (0.23, -0.23),
    (0.00, -0.33), (0.00, 0.33), (-0.33, 0.00), (0.33, 0.00),
    (-0.36, -0.36), (0.36, 0.36), (-0.36, 0.36), (0.36, -0.36),
]

#df_mesh = df_plot.dropna(subset=["Sprite Path"]).copy()

TOP_N_POR_TIPO_GENERACION = 3   

df_mesh = (
    df_plot
    .dropna(subset=["Sprite Path"])
    .sort_values(["Type 1", "Generation", "promedio_estadisticas"], ascending=[True, True, False])
    .groupby(["Type 1", "Generation"], group_keys=False)
    .head(TOP_N_POR_TIPO_GENERACION)
    .copy()
)

tipos_mesh = sorted(df_mesh["Type 1"].unique())
mapa_tipo_mesh = {tipo: i for i, tipo in enumerate(tipos_mesh)}
df_mesh["tipo_index"] = df_mesh["Type 1"].map(mapa_tipo_mesh)

df_mesh["pos_en_celda"] = df_mesh.groupby(["Type 1", "Generation"]).cumcount()
df_mesh["offset_x"] = df_mesh["pos_en_celda"].apply(lambda i: offsets[i % len(offsets)][0])
df_mesh["offset_y"] = df_mesh["pos_en_celda"].apply(lambda i: offsets[i % len(offsets)][1])
df_mesh["x_plot"] = df_mesh["Generation"] + df_mesh["offset_x"]
df_mesh["y_plot"] = df_mesh["tipo_index"] + df_mesh["offset_y"]

print("Pokémon dibujados como sprite en la malla 3D:", len(df_mesh))

rango_z = df_mesh["promedio_estadisticas"].max() - df_mesh["promedio_estadisticas"].min()
rango_x = df_mesh["Generation"].max() - df_mesh["Generation"].min()

SPRITE_ALTO_Z = max(rango_z * 0.07, 2)
SPRITE_ANCHO_X = max(rango_x * 0.04, 0.12)

print("SPRITE_ALTO_Z:", round(SPRITE_ALTO_Z, 2), "| SPRITE_ANCHO_X:", round(SPRITE_ANCHO_X, 3))

@lru_cache(maxsize=2000)
def cargar_sprite_array(ruta, size=18):
    try:
        img = Image.open(ruta).convert("RGBA")
        bbox = img.getbbox()
        if bbox:
            img = img.crop(bbox)
        img = img.resize((size, size), Image.Resampling.NEAREST)
        return np.array(img)
    except Exception as e:
        print(f"No se pudo procesar sprite {ruta}: {e}")
        return None

xs, ys, zs = [], [], []
ii, jj, kk = [], [], []
facecolors = []
indice = 0

for _, fila in df_mesh.iterrows():
    arr = cargar_sprite_array(fila["Sprite Path"], size=PIXEL_SPRITE)
    if arr is None:
        continue

    centro_x = float(fila["x_plot"])
    centro_y = float(fila["y_plot"])
    centro_z = float(fila["promedio_estadisticas"])

    alto, ancho, _ = arr.shape
    pixel_w = SPRITE_ANCHO_X / ancho
    pixel_h = SPRITE_ALTO_Z / alto

    for py in range(alto):
        for px in range(ancho):
            r, g, b, a = arr[py, px]
            if a < 40:
                continue

            x0 = centro_x - SPRITE_ANCHO_X / 2 + px * pixel_w
            x1 = x0 + pixel_w
            z1 = centro_z + SPRITE_ALTO_Z / 2 - py * pixel_h
            z0 = z1 - pixel_h
            y = centro_y

            xs.extend([x0, x1, x1, x0])
            ys.extend([y, y, y, y])
            zs.extend([z0, z0, z1, z1])

            ii.extend([indice, indice])
            jj.extend([indice + 1, indice + 2])
            kk.extend([indice + 2, indice + 3])

            color = f"rgba({r},{g},{b},{a / 255})"
            facecolors.extend([color, color])
            indice += 4

mesh_sprites = go.Mesh3d(
    x=xs, y=ys, z=zs, i=ii, j=jj, k=kk,
    facecolor=facecolors, flatshading=True, hoverinfo="skip",
    lighting=dict(ambient=1, diffuse=0, specular=0, roughness=1, fresnel=0),
    showscale=False, name="Sprites Pokémon"
)

fig_mesh = go.Figure()
fig_mesh.add_trace(mesh_sprites)

for tipo in tipos_mesh:
    data_tipo = df_mesh[df_mesh["Type 1"] == tipo]
    if data_tipo.empty:
        continue

    customdata = np.stack([
        data_tipo["Name"], data_tipo["Type 1"], data_tipo["ID"],
        data_tipo["promedio_estadisticas"], data_tipo["Generation"]
    ], axis=-1)

    fig_mesh.add_trace(
        go.Scatter3d(
            x=data_tipo["x_plot"], y=data_tipo["y_plot"], z=data_tipo["promedio_estadisticas"],
            mode="markers", name=tipo, customdata=customdata,
            marker=dict(size=9, color=color_por_tipo.get(tipo, "#CCCCCC"), opacity=0.13),
            hovertemplate=
                "<b style='font-size:15px'>%{customdata[0]}</b><br><br>" +
                "<b>Tipo:</b> %{customdata[1]}<br>" +
                "<b>Generación:</b> %{customdata[4]}<br>" +
                "<b>Promedio de estadísticas:</b> %{customdata[3]}<br>" +
                "<b>ID Pokédex:</b> %{customdata[2]}<br>" +
                "<extra></extra>"
        )
    )

z_min = int(np.floor(df_mesh["promedio_estadisticas"].min() / 10) * 10) - 10
z_max = int(np.ceil(df_mesh["promedio_estadisticas"].max() / 10) * 10) + 10
gen_min = int(df_mesh["Generation"].min())
gen_max = int(df_mesh["Generation"].max())

fig_mesh.update_layout(
    title=dict(
        text="Scatter 3D de Pokémon con sprites (todos)",
        x=0.02, y=0.95, font=dict(size=22, color="#22345b")
    ),
    width=1300, height=850, margin=dict(l=0, r=0, t=70, b=0),
    legend=dict(title="Tipo", x=0.86, y=0.85, bgcolor="rgba(255,255,255,0.88)",
                bordercolor="#d9e1ea", borderwidth=1),
    hoverlabel=dict(bgcolor="white", bordercolor="#cbd5e1", font=dict(size=13, color="#22345b")),
    scene=dict(
        xaxis=dict(
            title="Generación", tickmode="array",
            tickvals=list(range(gen_min, gen_max + 1)),
            ticktext=[str(i) if i < 9 else "Fan-Made" for i in range(gen_min, gen_max + 1)],
            range=[gen_min - 0.6, gen_max + 0.6],
            backgroundcolor="rgb(245,248,252)", gridcolor="white", zerolinecolor="white", showspikes=False
        ),
        yaxis=dict(
            title="Tipo", tickmode="array",
            tickvals=list(range(len(tipos_mesh))), ticktext=tipos_mesh,
            range=[len(tipos_mesh) - 0.7, -0.7],
            backgroundcolor="rgb(245,248,252)", gridcolor="white", zerolinecolor="white", showspikes=False
        ),
        zaxis=dict(
            title="Promedio de estadísticas", range=[z_min, z_max],
            backgroundcolor="rgb(245,248,252)", gridcolor="white", zerolinecolor="white", showspikes=False
        ),
        camera=dict(eye=dict(x=1.75, y=-2.25, z=0.95)),
        aspectmode="manual", aspectratio=dict(x=1.6, y=1.15, z=0.9)
    )
)

fig_mesh.show()

### Exportar la visualización final (con sprites) a HTML

In [ ]:
ruta_html_mesh = os.path.join(DATA_DIR, "pokemon_scatter3d_sprites.html")
fig_mesh.write_html(ruta_html_mesh, include_plotlyjs="cdn", full_html=True)
print(f"Visualización final (con sprites) exportada en: {ruta_html_mesh}")


## 16. Conclusiones

- **Tamaño final del dataset.** De los 1025 Pokémon descargados (generaciones 1 a 9), 988 quedaron con generación y tipo asignados; los ~37 restantes se descartaron por no tener una generación mapeada (formas especiales/regionales cuyo nombre no calza exactamente con el listado de `/generation/{n}`).

- **Distribución por generación.** Gen 1 (151), Gen 2 (100), Gen 3 (134), Gen 4 (104), Gen 5 (147), Gen 6 (66), Gen 7 (83), Gen 8 (89) y Gen 9 (114). Las generaciones 6 y 7 son las que menos Pokémon aportan a la muestra, mientras que Gen 1 y Gen 5 son las más numerosas.

- **Distribución por tipo principal.** Water es el tipo con más representantes (127), seguido de Normal (113) y Grass (102). En el otro extremo, Flying es el tipo principal menos común por mucho (solo 8), ya que casi siempre aparece como tipo secundario y no como principal.

- **Promedio de estadísticas por generación.** Hay una tendencia al alza con el tiempo: Gen 3 tiene el promedio más bajo (67.04), muy cerca de Gen 2 (67.86) y Gen 1 (67.94); Gen 9 tiene el promedio más alto (76.11), seguido de Gen 7 (75.29) y Gen 4 (73.67). Esto sugiere que, en promedio, los Pokémon de generaciones más recientes tienden a tener estadísticas base más altas que los de las primeras generaciones.

- **Cobertura de sprites para el gráfico 3D.** De los 988 Pokémon del dataset, solo 430 (~44%) lograron descargar su sprite en la corrida original, debido a que 20 descargas simultáneas saturaban la PokeAPI y cualquier falla de red se descartaba sin reintentar. Con los reintentos, el backoff y el registro de errores agregados después, se espera recuperar buena parte de ese ~56% que se perdía.

- **Pendiente por analizar.** Para completar el panorama haría falta calcular el promedio de estadísticas por tipo principal (`df.groupby("Type 1")["promedio_estadisticas"].mean()`) y comparar el promedio de los Pokémon legendarios/míticos contra el resto (`df.groupby("Legendary")["promedio_estadisticas"].mean()`), así como observar visualmente si los legendarios se agrupan en alguna zona particular del scatter 3D — ninguno de estos dos análisis está calculado todavía en el notebook.

- **Nota sobre la ejecución.** Antes de entregar la práctica, este notebook debe ejecutarse de corrido con *Kernel → Restart & Run All*, para que las celdas queden numeradas en orden secuencial (1, 2, 3, ...) y se confirme que no hay errores en ninguna celda.